> **Note:** This notebook requires a live OpenAI API key. Set `OPENAI_API_KEY` in your `.env` file before running. Smoke-run deferred — API key not available in CI.

# Tracing with LangSmith

Set `LANGCHAIN_TRACING_V2=true` and LangSmith **auto-attaches** to every LangChain run — including `create_agent` agents — with no manual callback wiring. Below we trace a plain chat model, then an agent.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "oreilly-llm-apps"

In [ ]:
from langchain_openai import ChatOpenAI

# With LANGCHAIN_TRACING_V2=true set above, every call is traced automatically.
llm = ChatOpenAI(model="gpt-5.5", use_responses_api=True)

llm.invoke("How to build an LLM app?")

In [ ]:
llm.invoke("Why did the chicken cross the street?")

## Tracing a `create_agent` run

Agents built with LangChain v1 `create_agent` are traced automatically too. Run the cell below and check your LangSmith project — the agent loop, tool calls, and final answer all appear as a single trace.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


agent = create_agent(
    "openai:gpt-5.5",
    tools=[add],
    system_prompt="You are a helpful math assistant.",
)

# This run is auto-traced to LangSmith (LANGCHAIN_TRACING_V2=true).
result = agent.invoke({"messages": [{"role": "user", "content": "What is 17 + 25?"}]})
print(result["messages"][-1].content)